# 01 — Exploratory Data Analysis

Reads `data/processed/features.parquet`. Per spec.md §9.1.

**Phase 7 v1 scope (validation pass).** Sections 1, 6, 8 plus the validation extras (§9 below) are filled in. Sections 2, 3, 4, 5, 7 are scaffolded — they layer in after modeling so the analytical narrative can use residuals + feature importances.

**Why this ordering.** The point of this validation pass is to catch silent bugs in Phase 4's feature engineering *before* committing to model interpretation. The Centrelink-day × SEIFA chart in §6 is the keystone of the augmentor story — if it's flat, modeling won't save it. The missingness map in §8 tells us whether each feature actually has signal across the project span.


## Setup


In [ ]:
from __future__ import annotations

import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fuel_pred import config

FEATURES_PATH = Path(config.DATA_PROCESSED) / "features.parquet"
if not FEATURES_PATH.exists():
    raise SystemExit(
        f"features.parquet not found at {FEATURES_PATH}.\n"
        f"Run `make features` (or `make all`) first."
    )

features = pd.read_parquet(FEATURES_PATH)
features["date"] = pd.to_datetime(features["date"]).dt.date
print(f"loaded {len(features):,} rows × {len(features.columns)} cols")
print(f"date range: {features['date'].min()} → {features['date'].max()}")
print(f"stations: {features['station_id'].nunique():,}")
print(f"fuels: {sorted(features['fuel_code'].unique().tolist())}")


In [ ]:
# Fold boundaries from spec §8.3.
FOLDS = [
    ("train",       config.SPAN_START,         config.TRAIN_END),
    ("val",         config.VAL_START,          config.VAL_END),
    ("test_normal", config.TEST_START,         config.TEST_NORMAL_END),
    ("test_crisis", config.TEST_CRISIS_START,  features['date'].max().isoformat()),
]

def fold_label(d: dt.date) -> str:
    """Map a date to its fold name per spec §8.3."""
    for name, start, end in FOLDS:
        if dt.date.fromisoformat(start) <= d <= dt.date.fromisoformat(end):
            return name
    return "out_of_span"

features["fold"] = features["date"].apply(fold_label)
features["fold"].value_counts().sort_index()


## 1. Dataset overview

Station count over time, fuel-code coverage, observation density. **Validation question:** does each fold have non-trivial data for both U91 and Diesel?


In [ ]:
# Active stations per month + fuel — spot-check coverage gaps.
monthly = (
    features.assign(month=pd.to_datetime(features["date"]).dt.to_period("M"))
    .groupby(["month", "fuel_code"], observed=True)["station_id"]
    .nunique()
    .unstack("fuel_code")
)
monthly.head(12)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(ax=ax, marker="o", linewidth=1.5)
ax.set_title("Active stations per month, by fuel")
ax.set_ylabel("distinct station_id")
ax.set_xlabel("")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Rows + observation density per fold + fuel.
fold_summary = (
    features.groupby(["fold", "fuel_code"], observed=True)
    .agg(
        rows=("date", "size"),
        stations=("station_id", "nunique"),
        date_min=("date", "min"),
        date_max=("date", "max"),
        non_null_price_pct=("price_mean", lambda s: 100 * s.notna().mean()),
    )
    .round(1)
)
fold_summary


## 2. Geographic distribution

Map of stations coloured by SA2 SEIFA, brand mix by region. *Deferred to post-modeling pass — not validation-critical.*


In [ ]:
# TODO (post-modeling): map (matplotlib or plotly), brand mix by SA2 SEIFA quintile.


## 3. Price level and dispersion

By fuel, by brand, over time. *Deferred to post-modeling pass — descriptive analytics.*


In [ ]:
# TODO (post-modeling): price distributions per fuel; ECDFs; brand boxplots.


## 4. The petrol cycle

ACF + FFT. *Deferred to post-modeling pass — see spec §13 Q4 for the optional sanity-check tiny-model.*


In [ ]:
# TODO (post-modeling): ACF plot, FFT, sanity-check tiny model.


## 5. The 2026 crisis

Visible regime change in Brent + retail. *Deferred to post-modeling pass — needs `data/static/crisis_events.csv` annotations.*


In [ ]:
# TODO (post-modeling): dual-axis Brent vs retail U91; vertical lines for crisis events.


## 6. Centrelink-day check ⭐

**This is the augmentor-story chart and must be in the notebook.** Average U91 price residual (vs the station's 28-day rolling mean) by `cal_day_of_fortnight`, segmented by SEIFA quintile.

If the augmentor adds value, this chart should show meaningfully *different* fortnight-pattern shapes across SEIFA quintiles — i.e. low-SEIFA SA2s should dip / spike on Centrelink Wednesdays and Thursdays differently from high-SEIFA SA2s. If the chart is flat (all quintiles overlap), the SA2 signal isn't there in our slice and Model B won't beat Model A meaningfully.

Anchor for `cal_day_of_fortnight = 0` is `config.DOF_ANCHOR` (2016-07-04, a Monday).


In [ ]:
# Restrict to U91 (the target fuel) and rows with a populated SA2 score.
u91 = features[
    (features["fuel_code"] == "U91")
    & features["price_mean"].notna()
    & features["roll_price_mean_28"].notna()
    & features["sa2_seifa_irsd_score"].notna()
].copy()

# Residual = today's price minus the station's past-28-day rolling mean.
# This is what isolates the within-cycle phase signal we're hoping the
# augmentor explains differently across SEIFA strata.
u91["residual"] = u91["price_mean"] - u91["roll_price_mean_28"]

# SEIFA quintile per row (low=disadvantaged, high=advantaged).
#  may collapse to <5 bins on small fixtures; in that
# case let pandas auto-label rather than fight the labels-mismatch error.
try:
    u91["seifa_q"] = pd.qcut(
        u91["sa2_seifa_irsd_score"], q=5,
        labels=["Q1 (most disadvantaged)", "Q2", "Q3", "Q4", "Q5 (most advantaged)"],
        duplicates="drop",
    )
except ValueError:
    u91["seifa_q"] = pd.qcut(u91["sa2_seifa_irsd_score"], q=5, duplicates="drop")
print(f"rows: {len(u91):,} (U91 with non-null roll_28 + SEIFA)")
u91["seifa_q"].value_counts().sort_index()


In [ ]:
centrelink = (
    u91.groupby(["cal_day_of_fortnight", "seifa_q"], observed=True)["residual"]
    .agg(["mean", "count"])
    .reset_index()
)
pivot_mean = centrelink.pivot(
    index="cal_day_of_fortnight", columns="seifa_q", values="mean"
)
pivot_mean


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for col in pivot_mean.columns:
    ax.plot(pivot_mean.index, pivot_mean[col], marker="o", linewidth=1.6, label=col)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xticks(range(0, 14))
ax.set_xlabel("cal_day_of_fortnight (0 = Mon anchored at 2016-07-04)")
ax.set_ylabel("Mean U91 residual (price − 28d rolling mean), c/L")
ax.set_title("Centrelink fortnight × SEIFA quintile — U91 price residual")
ax.legend(title="SEIFA quintile", loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Read this chart as:** if the lines diverge meaningfully across quintiles (especially around days 9-10 / 11-12 — Centrelink fortnightly payday windows), the augmentor block has signal. If the lines lie roughly on top of each other, the SA2 stratification isn't picking up a real pricing-behaviour difference — Model B should be expected to land close to Model A.

*Caveat for v1 small-data validation runs: with only a few months of data and a small station roster, sample size per cell is thin and the pattern may be noisy. The full `make features` run produces 1000s of rows per cell and the pattern (if real) should resolve cleanly.*


## 7. Cross-correlations

Brent (lagged) vs retail at Sydney metro vs regional. *Deferred to post-modeling pass.*


In [ ]:
# TODO (post-modeling): cross-correlation plots by metro/regional split.


## 8. Missingness map

Per-column null rate, broken down by feature block (lag/upstream/cal/ctx/stn/wx/sa2/y) and by fold. **Validation questions:**

- Are the SA2 columns the spec ships now (population, age, income, SEIFA) actually populated? (Phase 3 acceptance is ≥ 95%.)
- Are the 6 deferred SA2 derivations confirmed null? (Per spec §7.7.1.)
- Do the Phase 5 macro features populate across all folds, or do quarterly series have gaps?
- Do the lag features have the expected null cliff at the start of each station's history?


In [ ]:
# Block prefix → cluster of related columns.
BLOCK_PREFIXES = ("lag_", "roll_", "xfuel_", "upstream_", "cal_", "ctx_", "stn_", "wx_", "sa2_", "y_")

def block_of(col: str) -> str:
    for p in BLOCK_PREFIXES:
        if col.startswith(p):
            return p.rstrip("_")
    return "meta"

missing_overall = (
    pd.DataFrame({
        "col": features.columns,
        "block": [block_of(c) for c in features.columns],
        "null_pct": [100 * features[c].isna().mean() for c in features.columns],
    })
    .sort_values(["block", "null_pct"])
    .reset_index(drop=True)
)
missing_overall.groupby("block")["null_pct"].agg(["min", "mean", "max", "count"]).round(1)


In [ ]:
# SA2 column-by-column — explicit list per spec §7.7.
sa2_cols = [c for c in features.columns if c.startswith("sa2_")]
missing_overall[missing_overall["col"].isin(sa2_cols)][["col", "null_pct"]].round(1)


In [ ]:
# Phase 5 macro columns — verify population across folds.
phase5_cols = [
    "upstream_tgp_sydney_lag_0",
    "upstream_tgp_sydney_lag_7",
    "upstream_tgp_minus_brent_aud_lag_7",
    "ctx_cash_rate",
    "ctx_asx200_lag_1",
    "ctx_inflation_expectations_lag_7",
]
phase5_present = [c for c in phase5_cols if c in features.columns]
(
    features.groupby("fold", observed=True)[phase5_present]
    .apply(lambda d: 100 * d.isna().mean())
    .round(1)
)


In [ ]:
# Heatmap: % null per block per fold. Quick visual scan for surprise gaps.
rows = []
for fold_name in features["fold"].unique():
    fold_rows = features[features["fold"] == fold_name]
    if len(fold_rows) == 0:
        continue
    for block in BLOCK_PREFIXES:
        cols = [c for c in features.columns if c.startswith(block)]
        if not cols:
            continue
        rows.append({
            "fold": fold_name,
            "block": block.rstrip("_"),
            "null_pct": 100 * fold_rows[cols].isna().to_numpy().mean(),
        })
miss = pd.DataFrame(rows).pivot(index="block", columns="fold", values="null_pct").round(1)
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(miss.values, aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)
ax.set_xticks(range(len(miss.columns)))
ax.set_xticklabels(miss.columns, rotation=30, ha="right")
ax.set_yticks(range(len(miss.index)))
ax.set_yticklabels(miss.index)
for i in range(miss.shape[0]):
    for j in range(miss.shape[1]):
        v = miss.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}%", ha="center", va="center", fontsize=9,
                    color="black" if v < 50 else "white")
ax.set_title("% null per feature block × fold")
plt.colorbar(im, ax=ax, label="% null")
plt.tight_layout()
plt.show()


## 9. Validation extras

Three more checks specifically aimed at catching silent feature-engineering bugs before modeling.


### 9a. Lag feature sanity

`lag_price_1` should be nearly identical to `price_mean` shifted back by one row within each `(station_id, fuel_code)` group. The distribution of `(price_mean − lag_price_1)` should be tight around zero with occasional ±5–30 c/L spikes (the petrol cycle).


In [ ]:
u91 = features[features["fuel_code"] == "U91"].copy()
delta = (u91["price_mean"] - u91["lag_price_1"]).dropna()
print(f"price_mean - lag_price_1 — n={len(delta):,}")
print(delta.describe(percentiles=[0.01, 0.5, 0.99]).round(2))
print()
print(f"abs delta > 50 c/L: {(delta.abs() > 50).sum():,} rows ({100*(delta.abs() > 50).mean():.2f}%)")


In [ ]:
# days_since_last_price_change should be near-zero on busy days, occasionally
# spike during cycle troughs. Look for anything implausible (e.g. > 90 days).
dslp = u91["days_since_last_price_change"].dropna()
print(f"days_since_last_price_change — n={len(dslp):,}")
print(dslp.describe(percentiles=[0.5, 0.9, 0.99]).round(1))
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(dslp.clip(upper=60), bins=60, edgecolor="none")
ax.set_xlabel("days since last price change (clipped at 60)")
ax.set_ylabel("rows")
ax.set_title("Distribution of `days_since_last_price_change`, U91")
plt.tight_layout()
plt.show()


### 9b. Cross-fuel coverage

`xfuel_dl_*` features join Diesel data onto U91 rows. If a station only sells U91 (no Diesel), these are all null on its U91 rows. Coverage tells us how much the cross-fuel block can contribute.


In [ ]:
u91_cross = features[features["fuel_code"] == "U91"]
xfuel_cols = [c for c in features.columns if c.startswith("xfuel_")]
pd.DataFrame({
    "col": xfuel_cols,
    "non_null_rate_pct": [100 * u91_cross[c].notna().mean() for c in xfuel_cols],
}).round(1)


In [ ]:
# Per-station coverage of xfuel_dl_price_lag_0: how many stations have any DL data?
per_station = (
    u91_cross.groupby("station_id", observed=True)["xfuel_dl_price_lag_0"]
    .apply(lambda s: 100 * s.notna().mean())
)
print(f"stations with > 0% DL coverage: {(per_station > 0).sum()} / {len(per_station)}")
print(f"stations with > 50% DL coverage: {(per_station > 50).sum()} / {len(per_station)}")
print()
print("per-station coverage distribution:")
print(per_station.describe(percentiles=[0.25, 0.5, 0.75]).round(1))


### 9c. SA2 variance

The augmentor block can only help if the SA2 features actually *vary* across our station roster. If every station sits in a similar high-income, low-disadvantage SA2 (e.g. all metro Sydney), the SA2 columns become near-constant and Model B has nothing to learn from them.


In [ ]:
# Per-station view (deduplicate on station_id since SA2 is station-static).
stations_view = features.drop_duplicates("station_id")
ship_now_sa2 = [
    "sa2_total_population",
    "sa2_median_age",
    "sa2_median_household_income_weekly",
    "sa2_seifa_irsd_score",
]
stations_view[ship_now_sa2].describe(percentiles=[0.1, 0.5, 0.9]).round(1)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, col in zip(axes.ravel(), ship_now_sa2, strict=False):
    s = stations_view[col].dropna()
    ax.hist(s, bins=20, edgecolor="none")
    ax.set_title(f"{col} — n={len(s)}, std={s.std():.1f}")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**If the histograms above are flat / near-constant**, the augmentor block can't differentiate stations along that dimension — Model B's per-block contribution from `sa2_*` will be small. **If they're broadly distributed**, the block has the potential to help. SEIFA IRSD is the headline; aim for visible spread across deciles 1–10 (scores ~600–1200).

*Validation pass complete.* Modeling proceeds in `02_modeling.ipynb`; deferred descriptive sections (2, 3, 4, 5, 7) get filled in afterwards once we have residuals + feature importances to layer in.
